In [1]:
import numpy as np

def read_proteus_mesh_3d(node_file, ele_file):
    # Read .node file
    with open(node_file, 'r') as f:
        lines = f.readlines()
    num_nodes = int(lines[0].strip().split()[0])
    nodes = []
    for i in range(1, num_nodes + 1):
        parts = lines[i].strip().split()
        x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
        nodes.append((x, y, z))
    nodes = np.array(nodes)

    # Read .ele file
    with open(ele_file, 'r') as f:
        lines = f.readlines()
    num_elements = int(lines[0].strip().split()[0])
    elements = []
    for i in range(1, num_elements + 1):
        parts = lines[i].strip().split()
        n1, n2, n3, n4 = int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
        elements.append((n1 - 1, n2 - 1, n3 - 1, n4 - 1))  # convert to 0-based indexing
    elements = np.array(elements)

    return nodes, elements


In [2]:
nodes_ref_2_file_path="../Theis/Inputs_less_Q/ref_2/hollow_cylinder.node" 
elements_ref_2_file_path="../Theis/Inputs_less_Q/ref_2/hollow_cylinder.ele" 


nodes_ref_2, ele_ref_2= read_proteus_mesh_3d(nodes_ref_2_file_path,elements_ref_2_file_path )

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def plot_mesh_3d(nodes, elements, alpha=0.15, edge_color='gray', figsize=(10, 8)):
    """
    Plot a 3D tetrahedral mesh using matplotlib.

    Parameters:
    - nodes: (N, 3) array of (x, y, z) node coordinates
    - elements: (M, 4) array of tetrahedral elements (indices into nodes)
    - alpha: transparency of the surface
    - edge_color: color of edges
    - figsize: size of the matplotlib figure
    """
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    for tet in elements:
        pts = nodes[list(tet)]
        faces = [
            [pts[0], pts[1], pts[2]],
            [pts[0], pts[1], pts[3]],
            [pts[0], pts[2], pts[3]],
            [pts[1], pts[2], pts[3]],
        ]
        tri = Poly3DCollection(faces, alpha=alpha, edgecolor=edge_color)
        ax.add_collection3d(tri)

    # Set the axis limits
    xlim = [np.min(nodes[:, 0]), np.max(nodes[:, 0])]
    ylim = [np.min(nodes[:, 1]), np.max(nodes[:, 1])]
    zlim = [np.min(nodes[:, 2]), np.max(nodes[:, 2])]
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("3D Tetrahedral Mesh")
    plt.tight_layout()
    plt.show()


In [4]:
#plot_mesh_3d(nodes_ref_2, ele_ref_2)

In [5]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_2/data_3d"
file_pattern = "ref_2_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_2 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_2[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


C:\Users\12254\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\12254\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [ ]:
import numpy as np
from scipy.spatial import cKDTree

# --- Extract pressure point coordinates and values at time 0.0 ---
time_key = "time_0.0"
pressure_data = pressure_data_by_time_ref_2[time_key]

p_coords = np.column_stack((pressure_data["x"], pressure_data["y"], pressure_data["z"]))
p_heads = pressure_data["pressure_head"]

# --- Build KDTree from mesh nodes ---
node_coords = nodes_ref_2
tree = cKDTree(node_coords)

# --- Query nearest node for each pressure point ---
dists, idxs = tree.query(p_coords)

# --- Gather results ---
nearest_results = []
for i, (p, head, idx, dist) in enumerate(zip(p_coords, p_heads, idxs, dists)):
    n = node_coords[idx]
    nearest_results.append({
        "p_coord": tuple(p),
        "pressure_head": head,
        "n_coord": tuple(n),
        "distance": dist
    })

# --- Print first few matches ---
print(f"{'Idx':>4} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 100)
for i, entry in enumerate(nearest_results[:10], 1):  # show more if needed
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    ph = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {ph:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


In [6]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_2
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_2 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_2.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_2[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_2.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


C:\Users\12254\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (2.2000, 2.0000, 2.0) |   0.000000 | (2.2000, 2.0000, 2.0) | 0.000000e+00
   2 | time_0.0 | (2.1998, 2.0100, 2.0) |   0.000000 | (2.1998, 2.0100, 2.0) | 5.756336e-05
   3 | time_0.0 | (2.1990, 2.0199, 2.0) |   0.000000 | (2.1990, 2.0199, 2.0) | 1.490005e-05
   4 | time_0.0 | (2.1978, 2.0298, 2.0) |   0.000000 | (2.1978, 2.0298, 2.0) | 3.487474e-05
   5 | time_0.0 | (2.1960, 2.0396, 2.0) |   0.000000 | (2.1960, 2.0396, 2.0) | 4.521499e-05
   6 | time_0.0 | (2.1938, 2.0494, 2.0) |   0.000000 | (2.1938, 2.0494, 2.0) | 5.092310e-05
   7 | time_0.0 | (2.1911, 2.0590, 2.0) |   0.000000 | (2.1911, 2.0590, 2.0) | 5.108434e-05
   8 | time_0.0 | (2.1879, 2.0684, 2.0) |   0.000000 | (2.1879, 2.0684, 2.0) | 3.873423e-05
   9 | time_0.0 | (2.1843, 2.0777, 2.0) |   0

In [7]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_2["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_2 = {}

for time_key, matches in nearest_by_time_ref_2.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_2[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }

# --- Preview for a chosen time ---
#preview_key = "time_0.15"
#if preview_key in drawdown_by_time:
#    ddata = drawdown_by_time[preview_key]
##    print(f"\n=== Drawdown Comparison at {preview_key} ===")
#    print(f"{'Idx':>4} | {'(x_n, y_n, z_n)':>30} | {'psi0':>8} | {'h_num':>8} | {'h_theis':>8} | {'s_num':>10} | {'s_theis':>10}")
#    print("-" * 100)
#    for i in range(min(10, len(ddata["coords"]))):
#        xn, yn, zn = ddata["coords"][i]
#        psi0 = ddata["psi0"][i]
##        h_num = ddata["numerical_head"][i]
#        h_theis = ddata["theis_head"][i]
#        s_num = ddata["numerical_drawdown"][i]
#        s_theis = ddata["theis_drawdown"][i]
#        print(f"{i+1:4d} | ({xn:.4f}, {yn:.4f}, {zn:.1f}) | {psi0:8.4f} | {h_num:8.4f} | {h_theis:8.4f} | {s_num:10.6f} | {s_theis:10.6f}")


In [8]:
import numpy as np

# Redefine L2 error functions based on pressure head directly
def compute_voronoi_volumes(nodes, elements):
    """
    Approximate Voronoi volumes using tetrahedral element contributions.
    Each node gets 1/4 of the volume of each tetrahedron it is part of.
    """
    volumes = np.zeros(len(nodes))
    for tet in elements:
        tet_nodes = nodes[np.array(tet)]
        a, b, c, d = tet_nodes
        v = np.abs(np.dot((a - d), np.cross((b - d), (c - d)))) / 6.0
        for i in tet:
            volumes[i] += v / 4.0
    return volumes

def compute_L2_error(ph_numerical, ph_exact, weights):
    diff_squared = (ph_numerical - ph_exact) ** 2
    exact_squared = ph_exact ** 2
    l2_abs = np.sqrt(np.sum(weights * diff_squared))
    ref_norm = np.sqrt(np.sum(weights * exact_squared))
    rel_l2 = l2_abs / ref_norm if ref_norm > 0 else np.nan
    return l2_abs, rel_l2



def evaluate_pressure_head_errors(theis_drawdown_by_time, nearest_by_time, nodes, elements):
    volumes = compute_voronoi_volumes(nodes, elements)
    coord_to_index = {tuple(coord): i for i, coord in enumerate(nodes)}

    abs_errors = {}
    rel_errors = {}

    for time_key, entries in nearest_by_time.items():
        if time_key not in theis_drawdown_by_time:
            continue

        coords = []
        ph_num = []
        ph_theis = []

        for entry, theo in zip(entries, theis_drawdown_by_time[time_key]):
            coord = tuple(entry["n_coord"])
            i = coord_to_index.get(coord, -1)
            if i >= 0:
                coords.append(coord)
                ph_num.append(entry["pressure_head"])
                ph_theis.append(theo["theis_head"])

        indices = [coord_to_index[c] for c in coords]
        weights = np.array([volumes[i] for i in indices])
        ph_num = np.array(ph_num)
        ph_theis = np.array(ph_theis)

        l2_abs, l2_rel = compute_L2_error(ph_num, ph_theis, weights)
        abs_errors[time_key] = l2_abs
        rel_errors[time_key] = l2_rel

    return abs_errors, rel_errors


def evaluate_pressure_head_errors(drawdown_by_time, nearest_by_time, nodes, elements):
    volumes = compute_voronoi_volumes(nodes, elements)
    coord_to_index = {tuple(coord): i for i, coord in enumerate(nodes)}

    abs_errors = {}
    rel_errors = {}

    for time_key, draw_data in drawdown_by_time.items():
        if time_key not in nearest_by_time:
            continue

        coords = draw_data["coords"]
        ph_theis = draw_data["theis_head"]
        ph_num = draw_data["numerical_head"]

        indices = [coord_to_index.get(tuple(c), -1) for c in coords]
        mask = np.array([i >= 0 for i in indices])
        valid_indices = [i for i in indices if i >= 0]

        if not valid_indices:
            continue

        weights = np.array([volumes[i] for i in valid_indices])
        ph_num = np.array(ph_num)[mask]
        ph_theis = np.array(ph_theis)[mask]

        l2_abs, l2_rel = compute_L2_error(ph_num, ph_theis, weights)
        abs_errors[time_key] = l2_abs
        rel_errors[time_key] = l2_rel

    return abs_errors, rel_errors


# Execute the function using current data
abs_l2_error_by_time_ref_2, rel_l2_error_by_time_ref_2 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_2, nearest_by_time_ref_2, nodes_ref_2, ele_ref_2
)


In [9]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_2.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_2.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_2.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   4.372791e-04 |   2.518403e-03
      1.00 |   7.234921e-04 |   4.165648e-03
      1.50 |   9.351616e-04 |   5.383107e-03
      2.00 |   1.132980e-03 |   6.520501e-03
      2.50 |   1.302825e-03 |   7.496675e-03
      3.00 |   1.448356e-03 |   8.332804e-03
      3.50 |   1.575530e-03 |   9.063232e-03
      4.00 |   1.688659e-03 |   9.712810e-03
      4.50 |   1.790055e-03 |   1.029487e-02
      5.00 |   1.881942e-03 |   1.082221e-02
      5.50 |   1.957533e-03 |   1.125584e-02
      6.00 |   2.022987e-03 |   1.163118e-02
      6.50 |   2.079480e-03 |   1.195502e-02
      7.00 |   2.129124e-03 |   1.223949e-02
      7.50 |   2.173355e-03 |   1.249286e-02
      8.00 |   2.212286e-03 |   1.271578e-02
      8.50 |   2.246374e-03 |   1.291088e-02
      9.00 |   2.276906e-03 |   1.308556e-02
      9.50 |   2.303430e-03 |   1.323723e-02
     10.00 |   2.327030e-03 |   1.337211e-02
     10.5

# ref_1


In [76]:
nodes_ref_1_file_path="../Theis/Inputs_less_Q/ref_1/hollow_cylinder.node" 
elements_ref_1_file_path="../Theis/Inputs_less_Q/ref_1/hollow_cylinder.ele" 


nodes_ref_1, ele_ref_1= read_proteus_mesh_3d(nodes_ref_1_file_path,elements_ref_1_file_path )

In [77]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_1/data_3d"
file_pattern = "ref_1_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_1 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_1[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


In [78]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_1
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_1 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_1.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_1[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_1.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (4.0000, 2.0000, 2.0) |   0.000000 | (4.0000, 2.0000, 2.0) | 0.000000e+00
   2 | time_0.0 | (3.9901, 2.1991, 2.0) |   0.000000 | (3.9901, 2.1991, 2.0) | 5.246285e-05
   3 | time_0.0 | (3.9603, 2.3963, 2.0) |   0.000000 | (3.9603, 2.3963, 2.0) | 4.563237e-05
   4 | time_0.0 | (3.9111, 2.5895, 2.0) |   0.000000 | (3.9111, 2.5895, 2.0) | 4.677086e-05
   5 | time_0.0 | (3.8430, 2.7769, 2.0) |   0.000000 | (3.8430, 2.7769, 2.0) | 5.646338e-05
   6 | time_0.0 | (3.7564, 2.9565, 2.0) |   0.000000 | (3.7564, 2.9565, 2.0) | 4.387435e-05
   7 | time_0.0 | (3.6525, 3.1266, 2.0) |   0.000000 | (3.6525, 3.1266, 2.0) | 4.597138e-05
   8 | time_0.0 | (3.5321, 3.2856, 2.0) |   0.000000 | (3.5321, 3.2856, 2.0) | 2.715870e-05
   9 | time_0.0 | (3.3965, 0.5683, 2.0) |   0

In [79]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_1["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_1 = {}

for time_key, matches in nearest_by_time_ref_1.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_1[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }



In [80]:
# Execute the function using current data
abs_l2_error_by_time_ref_1, rel_l2_error_by_time_ref_1 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_1, nearest_by_time_ref_1, nodes_ref_1, ele_ref_1
)

In [81]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_1.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_1.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_1.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   2.946427e-04 |   1.697737e-03
      1.00 |   5.125594e-04 |   2.952578e-03
      1.50 |   6.628619e-04 |   3.817491e-03
      2.00 |   8.264506e-04 |   4.758656e-03
      2.50 |   9.737531e-04 |   5.605837e-03
      3.00 |   1.095126e-03 |   6.303608e-03
      3.50 |   1.199572e-03 |   6.903861e-03
      4.00 |   1.291149e-03 |   7.429998e-03
      4.50 |   1.371414e-03 |   7.891009e-03
      5.00 |   1.443384e-03 |   8.304271e-03
      5.50 |   1.486547e-03 |   8.551794e-03
      6.00 |   1.515143e-03 |   8.715543e-03
      6.50 |   1.532253e-03 |   8.813247e-03
      7.00 |   1.541040e-03 |   8.863109e-03
      7.50 |   1.544754e-03 |   8.883837e-03
      8.00 |   1.543858e-03 |   8.878082e-03
      8.50 |   1.537844e-03 |   8.842931e-03
      9.00 |   1.528732e-03 |   8.790001e-03
      9.50 |   1.516173e-03 |   8.717280e-03
     10.00 |   1.501581e-03 |   8.632906e-03
     10.5

In [88]:
nodes_ref_1_5_file_path="../Theis/Inputs_less_Q/ref_1_5/hollow_cylinder.node" 
elements_ref_1_5_file_path="../Theis/Inputs_less_Q/ref_1_5/hollow_cylinder.ele" 


nodes_ref_1_5, ele_ref_1_5= read_proteus_mesh_3d(nodes_ref_1_5_file_path,elements_ref_1_5_file_path )

In [89]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_1_5/data_3d"
file_pattern = "ref_1_5_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_1_5 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_1_5[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


In [90]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_1_5
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_1_5 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_1_5.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_1_5[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_1_5.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (2.0000, 4.0000, 2.0) |   0.000000 | (2.0000, 4.0000, 2.0) | 0.000000e+00
   2 | time_0.0 | (1.4824, 3.9319, 2.0) |   0.000000 | (1.4824, 3.9319, 2.0) | 6.154947e-05
   3 | time_0.0 | (1.0000, 3.7321, 2.0) |   0.000000 | (1.0000, 3.7321, 2.0) | 4.919243e-05
   4 | time_0.0 | (0.5858, 3.4142, 2.0) |   0.000000 | (0.5858, 3.4142, 2.0) | 1.402243e-05
   5 | time_0.0 | (2.5176, 3.9319, 0.0) |   2.000000 | (2.5176, 3.9319, 0.0) | 6.154947e-05
   6 | time_0.0 | (2.0000, 4.0000, 0.0) |   2.000000 | (2.0000, 4.0000, 0.0) | 0.000000e+00
   7 | time_0.0 | (1.4824, 3.9319, 0.0) |   2.000000 | (1.4824, 3.9319, 0.0) | 6.154947e-05
   8 | time_0.0 | (1.0000, 3.7321, 0.0) |   2.000000 | (1.0000, 3.7321, 0.0) | 4.919243e-05
   9 | time_0.0 | (0.5858, 3.4142, 0.0) |   2

In [91]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_1_5["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_1_5 = {}

for time_key, matches in nearest_by_time_ref_1_5.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_1_5[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }



In [92]:
# Execute the function using current data
abs_l2_error_by_time_ref_1_5, rel_l2_error_by_time_ref_1_5 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_1_5, nearest_by_time_ref_1_5, nodes_ref_1_5, ele_ref_1_5
)

In [93]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_1_5.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_1_5.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_1_5.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   1.890000e-04 |   1.097729e-03
      1.00 |   2.518799e-04 |   1.462551e-03
      1.50 |   2.876965e-04 |   1.670139e-03
      2.00 |   3.284807e-04 |   1.906527e-03
      2.50 |   4.428067e-04 |   2.569649e-03
      3.00 |   5.337830e-04 |   3.097132e-03
      3.50 |   6.222206e-04 |   3.609789e-03
      4.00 |   7.031106e-04 |   4.078583e-03
      4.50 |   7.754657e-04 |   4.497811e-03
      5.00 |   8.467179e-04 |   4.910597e-03
      5.50 |   8.496552e-04 |   4.927181e-03
      6.00 |   8.341812e-04 |   4.837036e-03
      6.50 |   8.068994e-04 |   4.678471e-03
      7.00 |   7.754597e-04 |   4.495847e-03
      7.50 |   7.474963e-04 |   4.333423e-03
      8.00 |   7.204537e-04 |   4.176375e-03
      8.50 |   6.968453e-04 |   4.039269e-03
      9.00 |   6.706940e-04 |   3.887453e-03
      9.50 |   6.472830e-04 |   3.751548e-03
     10.00 |   6.256489e-04 |   3.625965e-03
     10.5

In [26]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_0_5.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_0.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_0.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   3.062587e-04 |   1.767785e-03
      1.00 |   4.824338e-04 |   2.783952e-03
      1.50 |   5.963519e-04 |   3.440530e-03
      2.00 |   7.203521e-04 |   4.155095e-03
      2.50 |   8.563901e-04 |   4.938927e-03
      3.00 |   9.714350e-04 |   5.601558e-03
      3.50 |   1.070628e-03 |   6.172696e-03
      4.00 |   1.159255e-03 |   6.682864e-03
      4.50 |   1.240842e-03 |   7.152405e-03
      5.00 |   1.314862e-03 |   7.578301e-03
      5.50 |   1.343566e-03 |   7.743015e-03
      6.00 |   1.355415e-03 |   7.810626e-03
      6.50 |   1.354534e-03 |   7.804920e-03
      7.00 |   1.345658e-03 |   7.753194e-03
      7.50 |   1.334645e-03 |   7.689193e-03
      8.00 |   1.323033e-03 |   7.621785e-03
      8.50 |   1.307913e-03 |   7.534201e-03
      9.00 |   1.290808e-03 |   7.435221e-03
      9.50 |   1.274281e-03 |   7.339598e-03
     10.00 |   1.256294e-03 |   7.235605e-03
     10.5

## ref_0 

In [82]:
nodes_ref_0_file_path="../Theis/Inputs_less_Q/ref_0/hollow_cylinder.node" 
elements_ref_0_file_path="../Theis/Inputs_less_Q/ref_0/hollow_cylinder.ele" 


nodes_ref_0, ele_ref_0= read_proteus_mesh_3d(nodes_ref_0_file_path,elements_ref_0_file_path )

In [83]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_0/data_3d_1"
file_pattern = "ref_0_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_0 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_0[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


In [84]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_0
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_0 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_0.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_0[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_0.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (0.4825, 3.3027, 2.0) |   0.000000 | (0.4825, 3.3027, 2.0) | 4.512193e-05
   2 | time_0.0 | (0.2513, 2.9706, 2.0) |   0.000000 | (0.2513, 2.9706, 2.0) | 5.084664e-06
   3 | time_0.0 | (0.0917, 2.5987, 2.0) |   0.000000 | (0.0917, 2.5987, 2.0) | 2.625047e-05
   4 | time_0.0 | (0.0103, 2.2023, 2.0) |   0.000000 | (0.0103, 2.2023, 2.0) | 3.664568e-05
   5 | time_0.0 | (0.0103, 1.7977, 2.0) |   0.000000 | (0.0103, 1.7977, 2.0) | 3.664568e-05
   6 | time_0.0 | (0.7758, 3.5816, 0.0) |   2.000000 | (0.7758, 3.5816, 0.0) | 4.856590e-05
   7 | time_0.0 | (0.4825, 3.3027, 0.0) |   2.000000 | (0.4825, 3.3027, 0.0) | 4.512193e-05
   8 | time_0.0 | (0.2513, 2.9706, 0.0) |   2.000000 | (0.2513, 2.9706, 0.0) | 5.084664e-06
   9 | time_0.0 | (0.0917, 2.5987, 0.0) |   2

In [85]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_0["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_0 = {}

for time_key, matches in nearest_by_time_ref_0.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_0[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }



In [86]:
# Execute the function using current data
abs_l2_error_by_time_ref_0, rel_l2_error_by_time_ref_0 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_0, nearest_by_time_ref_0, nodes_ref_0, ele_ref_0
)

In [87]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_0.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_0.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_0.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   3.062587e-04 |   1.767785e-03
      1.00 |   4.824338e-04 |   2.783952e-03
      1.50 |   5.963519e-04 |   3.440530e-03
      2.00 |   7.203521e-04 |   4.155095e-03
      2.50 |   8.563901e-04 |   4.938927e-03
      3.00 |   9.714350e-04 |   5.601558e-03
      3.50 |   1.070628e-03 |   6.172696e-03
      4.00 |   1.159255e-03 |   6.682864e-03
      4.50 |   1.240842e-03 |   7.152405e-03
      5.00 |   1.314862e-03 |   7.578301e-03
      5.50 |   1.343566e-03 |   7.743015e-03
      6.00 |   1.355415e-03 |   7.810626e-03
      6.50 |   1.354534e-03 |   7.804920e-03
      7.00 |   1.345658e-03 |   7.753194e-03
      7.50 |   1.334645e-03 |   7.689193e-03
      8.00 |   1.323033e-03 |   7.621785e-03
      8.50 |   1.307913e-03 |   7.534201e-03
      9.00 |   1.290808e-03 |   7.435221e-03
      9.50 |   1.274281e-03 |   7.339598e-03
     10.00 |   1.256294e-03 |   7.235605e-03
     10.5

In [27]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_1.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_1.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_1.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   2.946427e-04 |   1.697737e-03
      1.00 |   5.125594e-04 |   2.952578e-03
      1.50 |   6.628619e-04 |   3.817491e-03
      2.00 |   8.264506e-04 |   4.758656e-03
      2.50 |   9.737531e-04 |   5.605837e-03
      3.00 |   1.095126e-03 |   6.303608e-03
      3.50 |   1.199572e-03 |   6.903861e-03
      4.00 |   1.291149e-03 |   7.429998e-03
      4.50 |   1.371414e-03 |   7.891009e-03
      5.00 |   1.443384e-03 |   8.304271e-03
      5.50 |   1.486547e-03 |   8.551794e-03
      6.00 |   1.515143e-03 |   8.715543e-03
      6.50 |   1.532253e-03 |   8.813247e-03
      7.00 |   1.541040e-03 |   8.863109e-03
      7.50 |   1.544754e-03 |   8.883837e-03
      8.00 |   1.543858e-03 |   8.878082e-03
      8.50 |   1.537844e-03 |   8.842931e-03
      9.00 |   1.528732e-03 |   8.790001e-03
      9.50 |   1.516173e-03 |   8.717280e-03
     10.00 |   1.501581e-03 |   8.632906e-03
     10.5

## ref_neg_1

In [29]:
nodes_ref_neg_1_file_path="../Theis/Inputs_less_Q/ref_neg_1/hollow_cylinder.node" 
elements_ref_neg_1_file_path="../Theis/Inputs_less_Q/ref_neg_1/hollow_cylinder.ele" 


nodes_ref_neg_1, ele_ref_neg_1= read_proteus_mesh_3d(nodes_ref_neg_1_file_path,elements_ref_neg_1_file_path )

In [31]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_neg_1/data_3d"
file_pattern = "ref_neg_1_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_neg_1 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_neg_1[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


In [32]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_neg_1
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_neg_1 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_neg_1.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_neg_1[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_neg_1.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (4.0000, 2.0000, 2.0) |   0.000000 | (4.0000, 2.0000, 2.0) | 0.000000e+00
   2 | time_0.0 | (3.8478, 2.7654, 2.0) |   0.000000 | (3.8478, 2.7654, 2.0) | 5.266515e-05
   3 | time_0.0 | (1.2346, 0.1522, 2.0) |   0.000000 | (1.2346, 0.1522, 2.0) | 3.314846e-05
   4 | time_0.0 | (2.0000, 0.0000, 2.0) |   0.000000 | (2.0000, 0.0000, 2.0) | 4.440892e-16
   5 | time_0.0 | (2.7654, 0.1522, 2.0) |   0.000000 | (2.7654, 0.1522, 2.0) | 3.314846e-05
   6 | time_0.0 | (3.4142, 0.5858, 2.0) |   0.000000 | (3.4142, 0.5858, 2.0) | 1.402243e-05
   7 | time_0.0 | (3.8478, 1.2346, 2.0) |   0.000000 | (3.8478, 1.2346, 2.0) | 5.266515e-05
   8 | time_0.0 | (4.0000, 2.0000, 0.0) |   2.000000 | (4.0000, 2.0000, 0.0) | 0.000000e+00
   9 | time_0.0 | (1.2346, 0.1522, 0.0) |   2

In [33]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_neg_1["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_neg_1 = {}

for time_key, matches in nearest_by_time_ref_neg_1.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_neg_1[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }



In [34]:
# Execute the function using current data
abs_l2_error_by_time_ref_neg_1, rel_l2_error_by_time_ref_neg_1 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_neg_1, nearest_by_time_ref_neg_1, nodes_ref_neg_1, ele_ref_neg_1
)

In [35]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_neg_1.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_neg_1.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_neg_1.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   3.169126e-04 |   1.842260e-03
      1.00 |   3.714936e-04 |   2.158972e-03
      1.50 |   3.829394e-04 |   2.224988e-03
      2.00 |   3.682861e-04 |   2.139437e-03
      2.50 |   3.984461e-04 |   2.314257e-03
      3.00 |   4.491421e-04 |   2.608329e-03
      3.50 |   5.135450e-04 |   2.981952e-03
      4.00 |   5.795817e-04 |   3.365008e-03
      4.50 |   6.465130e-04 |   3.753206e-03
      5.00 |   7.101335e-04 |   4.122141e-03
      5.50 |   7.001503e-04 |   4.063827e-03
      6.00 |   6.760297e-04 |   3.923499e-03
      6.50 |   6.445547e-04 |   3.740535e-03
      7.00 |   6.084933e-04 |   3.531003e-03
      7.50 |   5.831412e-04 |   3.383657e-03
      8.00 |   5.628108e-04 |   3.265480e-03
      8.50 |   5.414488e-04 |   3.141343e-03
      9.00 |   5.213661e-04 |   3.024653e-03
      9.50 |   5.027324e-04 |   2.916390e-03
     10.00 |   4.846703e-04 |   2.811462e-03
     10.5

In [41]:
nodes_ref_neg_2_file_path="../Theis/Inputs_less_Q/ref_neg_2/hollow_cylinder.node" 
elements_ref_neg_2_file_path="../Theis/Inputs_less_Q/ref_neg_2/hollow_cylinder.ele" 
nodes_ref_neg_2, ele_ref_neg_2= read_proteus_mesh_3d(nodes_ref_neg_2_file_path,elements_ref_neg_2_file_path )


In [37]:
import os
import glob
import pandas as pd

# Path to the folder and pattern
data_folder = "../Theis/Inputs_less_Q/ref_neg_2/data_3d"
file_pattern = "ref_neg_2_*.csv"

# Dictionary to store time step data
pressure_data_by_time_ref_neg_2 = {}

# Loop through all matching files
for file_path in sorted(glob.glob(os.path.join(data_folder, file_pattern))):
    df = pd.read_csv(file_path)

    if not all(col in df.columns for col in ["Time", "pressure_head", "Points:0", "Points:1", "Points:2"]):
        continue  # skip if required columns are missing

    # Extract time from the DataFrame (assumes all rows have the same time)
    time_val = float(df["Time"].iloc[0])
    timestep_key = f"time_{time_val:.1f}"

    # Extract data
    x = df["Points:0"].values
    y = df["Points:1"].values
    z = df["Points:2"].values
    pressure = df["pressure_head"].values

    # Store as dictionary entry
    pressure_data_by_time_ref_neg_2[timestep_key] = {
        "x": x,
        "y": y,
        "z": z,
        "pressure_head": pressure
    }

# Example: Access data at time 99.5
# print(pressure_data_by_time["time_99.5"]["pressure_head"])


In [38]:
import numpy as np
from scipy.spatial import cKDTree

# --- Build KDTree from mesh node coordinates ---
node_coords = nodes_ref_neg_2
tree = cKDTree(node_coords)

# --- Storage for all time step matches ---
nearest_by_time_ref_neg_2 = {}

# --- Loop over all time steps in pressure data ---
for time_key, data in pressure_data_by_time_ref_neg_2.items():
    p_coords = np.column_stack((data["x"], data["y"], data["z"]))
    p_heads = data["pressure_head"]

    # Query nearest node for each pressure point
    dists, idxs = tree.query(p_coords)

    matches = []
    for i in range(len(p_coords)):
        xp, yp, zp = p_coords[i]
        xn, yn, zn = node_coords[idxs[i]]
        head = p_heads[i]
        dist = dists[i]
        matches.append({
            "p_coord": (xp, yp, zp),
            "pressure_head": head,
            "n_coord": (xn, yn, zn),
            "distance": dist
        })

    # Store results for this time step
    nearest_by_time_ref_neg_2[time_key] = matches

# --- Preview: First few matches from time_0.0 ---
print(f"{'Idx':>4} | {'Time':>8} | {'(x_p, y_p, z_p)':>30} | {'pressure':>10} | {'(x_n, y_n, z_n)':>30} | {'Dist':>10}")
print("-" * 110)

preview = nearest_by_time_ref_neg_2.get("time_0.0", [])[:10]
for i, entry in enumerate(preview, 1):
    xp, yp, zp = entry["p_coord"]
    xn, yn, zn = entry["n_coord"]
    head = entry["pressure_head"]
    dist = entry["distance"]
    print(f"{i:4d} | {'time_0.0':>8} | ({xp:.4f}, {yp:.4f}, {zp:.1f}) | {head:10.6f} | "
          f"({xn:.4f}, {yn:.4f}, {zn:.1f}) | {dist:10.6e}")


 Idx |     Time |                (x_p, y_p, z_p) |   pressure |                (x_n, y_n, z_n) |       Dist
--------------------------------------------------------------------------------------------------------------
   1 | time_0.0 | (2.0000, 4.0000, 2.0) |   0.000000 | (2.0000, 4.0000, 2.0) | 0.000000e+00
   2 | time_0.0 | (0.5858, 3.4142, 2.0) |   0.000000 | (0.5858, 3.4142, 2.0) | 1.402243e-05
   3 | time_0.0 | (2.0000, 4.0000, 0.0) |   2.000000 | (2.0000, 4.0000, 0.0) | 0.000000e+00
   4 | time_0.0 | (0.5858, 3.4142, 0.0) |   2.000000 | (0.5858, 3.4142, 0.0) | 1.402243e-05
   5 | time_0.0 | (2.0000, 2.2000, 2.0) |   0.000000 | (2.0000, 2.2000, 2.0) | 0.000000e+00
   6 | time_0.0 | (1.8586, 2.1414, 2.0) |   0.000000 | (1.8586, 2.1414, 2.0) | 3.020228e-05
   7 | time_0.0 | (1.8000, 2.0000, 2.0) |   0.000000 | (1.8000, 2.0000, 2.0) | 0.000000e+00
   8 | time_0.0 | (2.0000, 2.2000, 0.0) |   2.000000 | (2.0000, 2.2000, 0.0) | 0.000000e+00
   9 | time_0.0 | (1.8586, 2.1414, 0.0) |   2

In [39]:
import numpy as np
from scipy.special import exp1

# --- Theis Parameters ---
Q = 1e-5       # pumping rate [m³/s]
T = 5e-4       # transmissivity [m²/s]
S = 1e-3       # storativity [-]
center = (2.0, 2.0)  # well location

# --- Initial pressure head (psi0) from time_0.0 ---
initial_heads_by_coord = {}
for entry in nearest_by_time_ref_neg_2["time_0.0"]:
    xn, yn, zn = entry["n_coord"]
    h0 = entry["pressure_head"]
    initial_heads_by_coord[(xn, yn, zn)] = h0

# --- Drawdown arrays per time step ---
drawdown_by_time_ref_neg_2 = {}

for time_key, matches in nearest_by_time_ref_neg_2.items():
    t_val = float(time_key.replace("time_", ""))
    if t_val == 0.0:
        continue  # skip initial condition

    coords = []
    theis_heads = []
    numerical_heads = []
    psi0_vals = []

    for entry in matches:
        xn, yn, zn = entry["n_coord"]
        coord = (xn, yn, zn)
        psi0 = initial_heads_by_coord.get(coord, None)
        if psi0 is None:
            continue  # skip if no initial head for this node

        # Compute Theis pressure head at this time
        r2 = (xn - center[0])**2 + (yn - center[1])**2
        u = (r2 * S) / (4.0 * T * t_val)
        h_theis = psi0 - (Q / (4.0 * np.pi * T)) * exp1(u)

        # Get numerical pressure head at this time
        h_num = entry["pressure_head"]

        # Collect all data
        coords.append(coord)
        psi0_vals.append(psi0)
        theis_heads.append(h_theis)
        numerical_heads.append(h_num)

    # Convert to arrays
    coords = np.array(coords)
    psi0_vals = np.array(psi0_vals)
    theis_heads = np.array(theis_heads)
    numerical_heads = np.array(numerical_heads)

    # Compute drawdowns
    theis_drawdown = psi0_vals - theis_heads
    numerical_drawdown = psi0_vals - numerical_heads

    # Store for this time
    drawdown_by_time_ref_neg_2[time_key] = {
        "coords": coords,
        #"psi0": psi0_vals,
        "theis_head": theis_heads,
        "numerical_head": numerical_heads,
        #"theis_drawdown": theis_drawdown,
        #"numerical_drawdown": numerical_drawdown
    }



In [42]:
# Execute the function using current data
abs_l2_error_by_time_ref_neg_2, rel_l2_error_by_time_ref_neg_2 = evaluate_pressure_head_errors(
    #theis_drawdown_by_time, nearest_by_time, nodes_ref_2, ele_ref_2
    drawdown_by_time_ref_neg_2, nearest_by_time_ref_neg_2, nodes_ref_neg_2, ele_ref_neg_2
)

In [43]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_neg_2.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_neg_2.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_neg_2.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   6.577836e-05 |   3.809025e-04
      1.00 |   3.136904e-05 |   1.815958e-04
      1.50 |   3.914503e-05 |   2.265640e-04
      2.00 |   1.377010e-04 |   7.968508e-04
      2.50 |   3.271467e-04 |   1.892860e-03
      3.00 |   4.188781e-04 |   2.423300e-03
      3.50 |   5.055225e-04 |   2.924219e-03
      4.00 |   5.871197e-04 |   3.395868e-03
      4.50 |   6.607239e-04 |   3.821227e-03
      5.00 |   7.262041e-04 |   4.199559e-03
      5.50 |   6.853181e-04 |   3.962800e-03
      6.00 |   6.527507e-04 |   3.774198e-03
      6.50 |   6.076619e-04 |   3.513248e-03
      7.00 |   5.643744e-04 |   3.262763e-03
      7.50 |   5.428983e-04 |   3.138411e-03
      8.00 |   5.260287e-04 |   3.040712e-03
      8.50 |   5.090076e-04 |   2.942158e-03
      9.00 |   4.940333e-04 |   2.855453e-03
      9.50 |   4.810077e-04 |   2.780027e-03
     10.00 |   4.678600e-04 |   2.703908e-03
     10.5

In [44]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_neg_1.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_neg_1.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_neg_1.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   3.169126e-04 |   1.842260e-03
      1.00 |   3.714936e-04 |   2.158972e-03
      1.50 |   3.829394e-04 |   2.224988e-03
      2.00 |   3.682861e-04 |   2.139437e-03
      2.50 |   3.984461e-04 |   2.314257e-03
      3.00 |   4.491421e-04 |   2.608329e-03
      3.50 |   5.135450e-04 |   2.981952e-03
      4.00 |   5.795817e-04 |   3.365008e-03
      4.50 |   6.465130e-04 |   3.753206e-03
      5.00 |   7.101335e-04 |   4.122141e-03
      5.50 |   7.001503e-04 |   4.063827e-03
      6.00 |   6.760297e-04 |   3.923499e-03
      6.50 |   6.445547e-04 |   3.740535e-03
      7.00 |   6.084933e-04 |   3.531003e-03
      7.50 |   5.831412e-04 |   3.383657e-03
      8.00 |   5.628108e-04 |   3.265480e-03
      8.50 |   5.414488e-04 |   3.141343e-03
      9.00 |   5.213661e-04 |   3.024653e-03
      9.50 |   5.027324e-04 |   2.916390e-03
     10.00 |   4.846703e-04 |   2.811462e-03
     10.5

In [45]:
# Sort the time keys numerically
sorted_keys = sorted(abs_l2_error_by_time_ref_0.keys(), key=lambda k: float(k.replace("time_", "")))

# Print header
print(f"{'Time':>10} | {'Rel L2 Error':>14} | {'Abs L2 Error':>14}")
print("-" * 45)

# Print sorted values
for key in sorted_keys:
    time_val = float(key.replace("time_", ""))
    rel = rel_l2_error_by_time_ref_0.get(key, np.nan)
    abs_ = abs_l2_error_by_time_ref_0.get(key, np.nan)
    print(f"{time_val:10.2f} | {rel:14.6e} | {abs_:14.6e}")


      Time |   Rel L2 Error |   Abs L2 Error
---------------------------------------------
      0.50 |   3.062587e-04 |   1.767785e-03
      1.00 |   4.824338e-04 |   2.783952e-03
      1.50 |   5.963519e-04 |   3.440530e-03
      2.00 |   7.203521e-04 |   4.155095e-03
      2.50 |   8.563901e-04 |   4.938927e-03
      3.00 |   9.714350e-04 |   5.601558e-03
      3.50 |   1.070628e-03 |   6.172696e-03
      4.00 |   1.159255e-03 |   6.682864e-03
      4.50 |   1.240842e-03 |   7.152405e-03
      5.00 |   1.314862e-03 |   7.578301e-03
      5.50 |   1.343566e-03 |   7.743015e-03
      6.00 |   1.355415e-03 |   7.810626e-03
      6.50 |   1.354534e-03 |   7.804920e-03
      7.00 |   1.345658e-03 |   7.753194e-03
      7.50 |   1.334645e-03 |   7.689193e-03
      8.00 |   1.323033e-03 |   7.621785e-03
      8.50 |   1.307913e-03 |   7.534201e-03
      9.00 |   1.290808e-03 |   7.435221e-03
      9.50 |   1.274281e-03 |   7.339598e-03
     10.00 |   1.256294e-03 |   7.235605e-03
     10.5

In [46]:
def compute_convergence_rate(error_dict_coarse, error_dict_fine, h_coarse, h_fine):
    rates_by_time = {}

    common_times = sorted(set(error_dict_coarse) & set(error_dict_fine),
                          key=lambda k: float(k.replace("time_", "")))

    for time_key in common_times:
        e_coarse = error_dict_coarse[time_key]
        e_fine = error_dict_fine[time_key]

        if e_coarse > 0 and e_fine > 0:
            rate = np.log(e_coarse / e_fine) / np.log(h_coarse / h_fine)
        else:
            rate = np.nan

        rates_by_time[time_key] = rate

    return rates_by_time


In [53]:
rates = compute_convergence_rate(
    abs_l2_error_by_time_ref_0,
    abs_l2_error_by_time_ref_1,
    h_coarse=0.8,  # for ref_neg_2
    h_fine=0.4    # for ref_2
)

print(f"{'Time':>10} | {'Conv. Rate':>12}")
print("-" * 26)
for key in sorted(rates, key=lambda k: float(k.replace("time_", ""))):
    t = float(key.replace("time_", ""))
    print(f"{t:10.2f} | {rates[key]:12.6f}")


      Time |   Conv. Rate
--------------------------
      0.50 |     0.058330
      1.00 |    -0.084841
      1.50 |    -0.149994
      2.00 |    -0.195673
      2.50 |    -0.182732
      3.00 |    -0.170350
      3.50 |    -0.161503
      4.00 |    -0.152895
      4.50 |    -0.141781
      5.00 |    -0.131979
      5.50 |    -0.143332
      6.00 |    -0.158153
      6.50 |    -0.175290
      7.00 |    -0.193022
      7.50 |    -0.208351
      8.00 |    -0.220119
      8.50 |    -0.231070
      9.00 |    -0.241488
      9.50 |    -0.248177
     10.00 |    -0.254733
     10.50 |    -0.257438
     11.00 |    -0.258415
     11.50 |    -0.256994
     12.00 |    -0.253926
     12.50 |    -0.247828
     13.00 |    -0.239176
     13.50 |    -0.229684
     14.00 |    -0.217425
     14.50 |    -0.204053
     15.00 |    -0.191641
     15.50 |    -0.172852
     16.00 |    -0.158111
     16.50 |    -0.141562
     17.00 |    -0.121697
     17.50 |    -0.102408
     18.00 |    -0.081745
     18.50 

In [51]:
rates = compute_convergence_rate(
    abs_l2_error_by_time_ref_neg_2,
    abs_l2_error_by_time_ref_neg_1,
    h_coarse=0.8,  # for ref_neg_2
    h_fine=0.4    # for ref_2
)

print(f"{'Time':>10} | {'Conv. Rate':>12}")
print("-" * 26)
for key in sorted(rates, key=lambda k: float(k.replace("time_", ""))):
    t = float(key.replace("time_", ""))
    print(f"{t:10.2f} | {rates[key]:12.6f}")


      Time |   Conv. Rate
--------------------------
      0.50 |    -2.273983
      1.00 |    -3.571541
      1.50 |    -3.295807
      2.00 |    -1.424850
      2.50 |    -0.289981
      3.00 |    -0.106153
      3.50 |    -0.028206
      4.00 |     0.013171
      4.50 |     0.025913
      5.00 |     0.026844
      5.50 |    -0.036319
      6.00 |    -0.055971
      6.50 |    -0.090439
      7.00 |    -0.113984
      7.50 |    -0.108549
      8.00 |    -0.102886
      8.50 |    -0.094507
      9.00 |    -0.083050
      9.50 |    -0.069085
     10.00 |    -0.056274
     10.50 |    -0.047792
     11.00 |    -0.046260
     11.50 |    -0.048326
     12.00 |    -0.040793
     12.50 |    -0.024841
     13.00 |     0.014913
     13.50 |     0.008063
     14.00 |     0.020054
     14.50 |     0.052239
     15.00 |     0.046900
     15.50 |     0.077144
     16.00 |     0.099219
     16.50 |     0.086496
     17.00 |     0.125023
     17.50 |     0.116449
     18.00 |     0.149704
     18.50 